# Fire Simple Extension 3 — Narrativa e Simulação em Python

Este notebook reproduz, em Python, a lógica do modelo **Fire Simple Extension 3**, inspirado no modelo de propagação de incêndio em floresta do NetLogo.

A simulação representa uma floresta em uma grade bidimensional, onde:

- cada célula pode estar vazia, conter uma árvore verde, estar em chamas ou já ter queimado;
- o fogo começa na borda esquerda do ambiente;
- a propagação ocorre para os quatro vizinhos diretos;
- a probabilidade de propagação é afetada pela direção e intensidade do vento;
- quando `big_jumps` está ativado, faíscas podem saltar para posições mais distantes, simulando transporte pelo vento.

O objetivo é observar como **densidade da floresta**, **probabilidade de propagação**, **vento** e **saltos de faíscas** alteram o comportamento global do incêndio.

## 1. Modelo original em NetLogo

O modelo original define uma variável global chamada `initial-trees`, responsável por registrar quantas árvores verdes existiam no início.

No procedimento `setup`, o ambiente é limpo, árvores são distribuídas aleatoriamente conforme a variável `density`, e a coluna da esquerda é incendiada.

No procedimento `go`, cada árvore em chamas verifica seus quatro vizinhos. Se houver árvores verdes, a chance de ignição é ajustada pelo vento:

- vento sul pode ajudar ou dificultar a propagação vertical;
- vento oeste pode ajudar ou dificultar a propagação horizontal;
- árvores queimadas ficam escurecidas;
- a simulação termina quando não há mais células vermelhas.

## 2. Estados da simulação

Nesta versão em Python, usaremos os seguintes valores:

| Valor | Estado |
|---:|---|
| 0 | célula vazia |
| 1 | árvore verde |
| 2 | árvore em chamas |
| 3 | árvore queimada |

A lógica é equivalente ao modelo NetLogo, mas adaptada para execução em notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import pandas as pd

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

## 3. Parâmetros do modelo

Os principais parâmetros são:

- `density`: porcentagem de células ocupadas por árvores;
- `probability_of_spread`: probabilidade base de propagação do fogo;
- `south_wind_speed`: intensidade do vento sul;
- `west_wind_speed`: intensidade do vento oeste;
- `big_jumps`: ativa ou desativa saltos de faíscas;
- `width` e `height`: dimensões da grade.

In [ ]:
width = 80
height = 60

density = 58
probability_of_spread = 55

south_wind_speed = 10
west_wind_speed = 15

big_jumps = True
max_ticks = 300

## 4. Função de inicialização

A função `setup()` cria a floresta.

Primeiro, cada célula recebe uma árvore com probabilidade definida por `density`. Depois, a primeira coluna da esquerda é marcada como fogo inicial.

In [ ]:
EMPTY = 0
TREE = 1
FIRE = 2
BURNED = 3

def setup(width, height, density, rng):
    '''
    Inicializa a grade da floresta.

    Args:
        width: largura da grade
        height: altura da grade
        density: percentual de ocupação por árvores
        rng: gerador aleatório do NumPy

    Returns:
        grid: matriz com estados da simulação
        initial_trees: quantidade inicial de árvores verdes
    '''
    grid = np.zeros((height, width), dtype=np.int8)

    tree_mask = rng.random((height, width)) < (density / 100)
    grid[tree_mask] = TREE

    # Coluna esquerda inicia em chamas
    grid[:, 0] = FIRE

    initial_trees = np.sum(grid == TREE)
    return grid, initial_trees

grid, initial_trees = setup(width, height, density, rng)
initial_trees

## 5. Visualização da floresta

As cores seguem a interpretação do modelo:

- branco: vazio;
- verde: árvore;
- vermelho: fogo;
- preto: queimado.

In [ ]:
cmap = ListedColormap(["white", "green", "red", "black"])

def plot_grid(grid, title="Estado da simulação"):
    plt.figure(figsize=(10, 7))
    plt.imshow(grid, cmap=cmap, vmin=0, vmax=3)
    plt.title(title)
    plt.axis("off")
    plt.show()

plot_grid(grid, "Estado inicial da floresta")

## 6. Regras de propagação

Cada célula em chamas avalia seus quatro vizinhos diretos: norte, sul, leste e oeste.

A probabilidade de propagação começa em `probability_of_spread` e é ajustada conforme a posição do fogo em relação à árvore vizinha.

A implementação abaixo segue a mesma interpretação do código NetLogo:

- se o fogo está ao norte da árvore, o vento sul dificulta;
- se o fogo está ao sul da árvore, o vento sul facilita;
- se o fogo está a leste da árvore, o vento oeste dificulta;
- se o fogo está a oeste da árvore, o vento oeste facilita.

In [ ]:
def step_fire(
    grid,
    probability_of_spread,
    south_wind_speed,
    west_wind_speed,
    big_jumps,
    rng
):
    '''
    Executa um tick da simulação.

    Returns:
        new_grid: grade atualizada
        ignitions: número de novas ignições
    '''
    height, width = grid.shape
    new_grid = grid.copy()
    ignitions = 0

    burning_cells = np.argwhere(grid == FIRE)

    for row, col in burning_cells:
        # (delta_linha, delta_coluna, direção do fogo em relação ao vizinho)
        neighbors = [
            (-1,  0, 180),  # árvore ao norte; fogo está ao sul dela
            ( 1,  0,   0),  # árvore ao sul; fogo está ao norte dela
            ( 0, -1,  90),  # árvore a oeste; fogo está a leste dela
            ( 0,  1, 270),  # árvore a leste; fogo está a oeste dela
        ]

        for dr, dc, direction in neighbors:
            nr, nc = row + dr, col + dc

            if 0 <= nr < height and 0 <= nc < width and grid[nr, nc] == TREE:
                probability = probability_of_spread

                if direction == 0:
                    probability -= south_wind_speed
                elif direction == 90:
                    probability -= west_wind_speed
                elif direction == 180:
                    probability += south_wind_speed
                elif direction == 270:
                    probability += west_wind_speed

                probability = max(0, min(100, probability))

                if rng.integers(0, 100) < probability:
                    new_grid[nr, nc] = FIRE
                    ignitions += 1

                    if big_jumps:
                        jump_col = nc + int(round(west_wind_speed / 5))
                        jump_row = nr + int(round(south_wind_speed / 5))

                        if (
                            0 <= jump_row < height
                            and 0 <= jump_col < width
                            and grid[jump_row, jump_col] == TREE
                        ):
                            new_grid[jump_row, jump_col] = FIRE
                            ignitions += 1

        new_grid[row, col] = BURNED

    return new_grid, ignitions

## 7. Execução da simulação

A simulação avança até que não haja mais células em chamas ou até atingir `max_ticks`.

In [ ]:
def run_simulation(
    width=80,
    height=60,
    density=58,
    probability_of_spread=55,
    south_wind_speed=10,
    west_wind_speed=15,
    big_jumps=True,
    max_ticks=300,
    seed=42
):
    rng = np.random.default_rng(seed)
    grid, initial_trees = setup(width, height, density, rng)

    history = []
    snapshots = {}
    tick = 0

    while np.any(grid == FIRE) and tick < max_ticks:
        trees = np.sum(grid == TREE)
        burning = np.sum(grid == FIRE)
        burned = np.sum(grid == BURNED)

        history.append({
            "tick": tick,
            "trees": trees,
            "burning": burning,
            "burned": burned,
            "percent_burned": 100 * burned / max(initial_trees, 1)
        })

        if tick in [0, 5, 10, 20, 40, 80, 120]:
            snapshots[tick] = grid.copy()

        grid, ignitions = step_fire(
            grid,
            probability_of_spread,
            south_wind_speed,
            west_wind_speed,
            big_jumps,
            rng
        )

        tick += 1

    history.append({
        "tick": tick,
        "trees": np.sum(grid == TREE),
        "burning": np.sum(grid == FIRE),
        "burned": np.sum(grid == BURNED),
        "percent_burned": 100 * np.sum(grid == BURNED) / max(initial_trees, 1)
    })

    snapshots[tick] = grid.copy()

    return grid, pd.DataFrame(history), snapshots, initial_trees

final_grid, history, snapshots, initial_trees = run_simulation(
    width=width,
    height=height,
    density=density,
    probability_of_spread=probability_of_spread,
    south_wind_speed=south_wind_speed,
    west_wind_speed=west_wind_speed,
    big_jumps=big_jumps,
    max_ticks=max_ticks,
    seed=RANDOM_SEED
)

history.tail()

## 8. Resultado final

A figura abaixo mostra o estado final da simulação.

In [ ]:
plot_grid(final_grid, "Estado final da simulação")

## 9. Evolução temporal

Agora observamos a quantidade de árvores verdes, árvores em chamas e árvores queimadas ao longo do tempo.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history["tick"], history["trees"], label="Árvores verdes")
plt.plot(history["tick"], history["burning"], label="Em chamas")
plt.plot(history["tick"], history["burned"], label="Queimadas")
plt.xlabel("Tick")
plt.ylabel("Quantidade de células")
plt.title("Evolução temporal da simulação")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history["tick"], history["percent_burned"])
plt.xlabel("Tick")
plt.ylabel("Percentual queimado (%)")
plt.title("Percentual acumulado de árvores queimadas")
plt.grid(True, alpha=0.3)
plt.show()

## 10. Snapshots da propagação

A sequência abaixo mostra alguns momentos da propagação do incêndio.

In [ ]:
for tick, snapshot in snapshots.items():
    plot_grid(snapshot, f"Snapshot — tick {tick}")

## 11. Métricas finais

As métricas finais ajudam a interpretar o comportamento global da simulação.

In [ ]:
final_metrics = {
    "árvores_iniciais": initial_trees,
    "árvores_restantes": int(np.sum(final_grid == TREE)),
    "árvores_queimadas": int(np.sum(final_grid == BURNED)),
    "percentual_queimado": float(100 * np.sum(final_grid == BURNED) / max(initial_trees, 1)),
    "ticks_executados": int(history["tick"].max()),
    "densidade": density,
    "probabilidade_base": probability_of_spread,
    "vento_sul": south_wind_speed,
    "vento_oeste": west_wind_speed,
    "big_jumps": big_jumps
}

pd.DataFrame([final_metrics])

## 12. Experimentos comparativos

Podemos comparar diferentes densidades da floresta para observar quando o fogo tende a atravessar grande parte do ambiente.

Essa análise se aproxima da ideia de **limiar crítico** ou **percolação**, comum em modelos de sistemas complexos.

In [ ]:
densities = [35, 45, 55, 65, 75]
results = []

for d in densities:
    final_grid_d, history_d, snapshots_d, initial_trees_d = run_simulation(
        width=width,
        height=height,
        density=d,
        probability_of_spread=probability_of_spread,
        south_wind_speed=south_wind_speed,
        west_wind_speed=west_wind_speed,
        big_jumps=big_jumps,
        max_ticks=max_ticks,
        seed=RANDOM_SEED
    )

    results.append({
        "density": d,
        "initial_trees": initial_trees_d,
        "burned": int(np.sum(final_grid_d == BURNED)),
        "remaining_trees": int(np.sum(final_grid_d == TREE)),
        "percent_burned": 100 * np.sum(final_grid_d == BURNED) / max(initial_trees_d, 1),
        "ticks": int(history_d["tick"].max())
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(results_df["density"], results_df["percent_burned"], marker="o")
plt.xlabel("Densidade inicial de árvores (%)")
plt.ylabel("Percentual queimado (%)")
plt.title("Efeito da densidade na propagação do fogo")
plt.grid(True, alpha=0.3)
plt.show()

## 13. Interpretação

A simulação evidencia que a propagação do fogo não depende apenas da presença de árvores, mas da conectividade espacial entre elas.

Em densidades baixas, o fogo encontra barreiras naturais, pois há muitas células vazias. Em densidades mais altas, a floresta se torna mais conectada, permitindo que o fogo atravesse grandes regiões.

O vento altera a assimetria da propagação, favorecendo algumas direções e dificultando outras. Quando `big_jumps` está ativado, a propagação pode ultrapassar pequenas barreiras, pois faíscas conseguem incendiar árvores mais distantes.

Esse tipo de modelo é útil para discutir:

- sistemas complexos;
- autômatos celulares;
- dinâmica espacial;
- limiares críticos;
- propagação em redes;
- simulações baseadas em agentes.

## 14. Próximas extensões possíveis

Algumas melhorias possíveis para pesquisa ou ensino:

1. incluir umidade do solo ou da vegetação;
2. variar o vento ao longo do tempo;
3. diferenciar tipos de vegetação;
4. medir se o fogo atravessou de um lado ao outro da grade;
5. repetir a simulação várias vezes por cenário;
6. calcular média, desvio padrão e intervalo de confiança;
7. exportar os resultados em CSV;
8. criar animação da propagação.

# 15. Versão estendida do modelo

Nesta etapa, adicionamos melhorias para uso em pesquisa e ensino:

1. umidade do solo ou da vegetação;
2. vento variável ao longo do tempo;
3. diferentes tipos de vegetação;
4. verificação se o fogo atravessou a grade da esquerda para a direita;
5. repetição da simulação várias vezes por cenário;
6. cálculo de média, desvio padrão e intervalo de confiança;
7. exportação dos resultados em CSV;
8. criação de animação da propagação.

A partir daqui, o modelo deixa de ser apenas uma reprodução simples do NetLogo e passa a funcionar como uma pequena plataforma experimental.

## 15.1 Novos estados e tipos de vegetação

Além dos estados básicos da célula, vamos criar uma matriz separada chamada `vegetation`.

Cada árvore poderá ter um tipo:

| Valor | Tipo de vegetação | Característica |
|---:|---|---|
| 0 | sem vegetação | célula vazia |
| 1 | grama ou vegetação baixa | queima com mais facilidade |
| 2 | arbusto | comportamento intermediário |
| 3 | árvore densa | pode queimar mais intensamente |

A umidade será representada em uma matriz chamada `moisture`, com valores entre 0 e 1. Quanto maior a umidade, menor a chance de ignição.

In [ ]:
# Tipos de vegetação
NO_VEG = 0
GRASS = 1
SHRUB = 2
DENSE_TREE = 3

vegetation_names = {
    NO_VEG: "sem vegetação",
    GRASS: "grama/baixa",
    SHRUB: "arbusto",
    DENSE_TREE: "árvore densa"
}

# Fatores de inflamabilidade
# Valores maiores aumentam a probabilidade de propagação.
flammability = {
    GRASS: 1.20,
    SHRUB: 1.00,
    DENSE_TREE: 0.85
}

# Duração do fogo por tipo de vegetação
# Isso permite que vegetações mais densas permaneçam queimando por mais tempo.
burn_duration = {
    GRASS: 1,
    SHRUB: 2,
    DENSE_TREE: 3
}

## 15.2 Inicialização estendida

A função abaixo cria:

- a grade de estados;
- a matriz de tipos de vegetação;
- a matriz de umidade;
- a matriz de idade do fogo, usada para controlar por quantos ticks uma célula permanece em chamas.

A coluna esquerda inicia em chamas, como no modelo original.

In [ ]:
def setup_extended(
    width,
    height,
    density,
    mean_moisture=0.25,
    moisture_variation=0.10,
    vegetation_probs=(0.35, 0.40, 0.25),
    seed=42
):
    rng = np.random.default_rng(seed)

    grid = np.zeros((height, width), dtype=np.int8)
    vegetation = np.zeros((height, width), dtype=np.int8)
    moisture = np.zeros((height, width), dtype=float)
    fire_age = np.zeros((height, width), dtype=np.int8)

    tree_mask = rng.random((height, width)) < (density / 100)

    grid[tree_mask] = TREE

    vegetation_choices = rng.choice(
        [GRASS, SHRUB, DENSE_TREE],
        size=(height, width),
        p=vegetation_probs
    )
    vegetation[tree_mask] = vegetation_choices[tree_mask]

    moisture_values = rng.normal(mean_moisture, moisture_variation, size=(height, width))
    moisture = np.clip(moisture_values, 0, 1)
    moisture[~tree_mask] = 0

    # Ignição inicial na borda esquerda
    grid[:, 0] = FIRE
    vegetation[:, 0] = np.where(vegetation[:, 0] == NO_VEG, SHRUB, vegetation[:, 0])
    moisture[:, 0] = np.minimum(moisture[:, 0], 0.20)

    initial_trees = int(np.sum(grid == TREE))

    return grid, vegetation, moisture, fire_age, initial_trees, rng

## 15.3 Vento variável ao longo do tempo

Agora o vento pode mudar a cada tick.

Usaremos uma função senoidal simples para representar variação temporal, acrescida de ruído aleatório. Isso permite simular rajadas e mudanças graduais.

In [ ]:
def dynamic_wind(tick, base_south=10, base_west=15, amplitude=8, noise=2, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    south = base_south + amplitude * np.sin(2 * np.pi * tick / 40) + rng.normal(0, noise)
    west = base_west + amplitude * np.cos(2 * np.pi * tick / 55) + rng.normal(0, noise)

    south = float(np.clip(south, -30, 30))
    west = float(np.clip(west, -30, 30))

    return south, west

## 15.4 Regra estendida de propagação

A probabilidade de ignição passa a considerar:

- probabilidade base;
- direção e intensidade do vento;
- tipo de vegetação;
- umidade local;
- possibilidade de faíscas saltarem quando `big_jumps=True`.

A fórmula usada é uma aproximação didática:

`probabilidade final = probabilidade ajustada pelo vento × inflamabilidade × efeito da umidade`

O efeito da umidade reduz a chance de propagação.

In [ ]:
def step_fire_extended(
    grid,
    vegetation,
    moisture,
    fire_age,
    probability_of_spread,
    south_wind_speed,
    west_wind_speed,
    big_jumps,
    rng
):
    height, width = grid.shape

    new_grid = grid.copy()
    new_fire_age = fire_age.copy()
    ignitions = 0

    burning_cells = np.argwhere(grid == FIRE)

    for row, col in burning_cells:
        neighbors = [
            (-1,  0, 180),
            ( 1,  0,   0),
            ( 0, -1,  90),
            ( 0,  1, 270),
        ]

        for dr, dc, direction in neighbors:
            nr, nc = row + dr, col + dc

            if 0 <= nr < height and 0 <= nc < width and grid[nr, nc] == TREE:
                probability = probability_of_spread

                if direction == 0:
                    probability -= south_wind_speed
                elif direction == 90:
                    probability -= west_wind_speed
                elif direction == 180:
                    probability += south_wind_speed
                elif direction == 270:
                    probability += west_wind_speed

                veg_type = int(vegetation[nr, nc])
                veg_factor = flammability.get(veg_type, 1.0)

                # Umidade reduz a probabilidade.
                # Com moisture = 0, não reduz.
                # Com moisture = 1, reduz fortemente.
                moisture_factor = 1 - 0.75 * moisture[nr, nc]

                probability = probability * veg_factor * moisture_factor
                probability = max(0, min(100, probability))

                if rng.integers(0, 100) < probability:
                    new_grid[nr, nc] = FIRE
                    new_fire_age[nr, nc] = 0
                    ignitions += 1

                    if big_jumps:
                        jump_col = nc + int(round(west_wind_speed / 5))
                        jump_row = nr + int(round(south_wind_speed / 5))

                        if (
                            0 <= jump_row < height
                            and 0 <= jump_col < width
                            and grid[jump_row, jump_col] == TREE
                        ):
                            jump_veg_type = int(vegetation[jump_row, jump_col])
                            jump_moisture_factor = 1 - 0.75 * moisture[jump_row, jump_col]
                            jump_probability = probability_of_spread * flammability.get(jump_veg_type, 1.0) * jump_moisture_factor
                            jump_probability = max(0, min(100, jump_probability))

                            if rng.integers(0, 100) < jump_probability:
                                new_grid[jump_row, jump_col] = FIRE
                                new_fire_age[jump_row, jump_col] = 0
                                ignitions += 1

        veg_current = int(vegetation[row, col])
        duration = burn_duration.get(veg_current, 1)

        if fire_age[row, col] + 1 >= duration:
            new_grid[row, col] = BURNED
            new_fire_age[row, col] = 0
        else:
            new_grid[row, col] = FIRE
            new_fire_age[row, col] = fire_age[row, col] + 1

    return new_grid, new_fire_age, ignitions

## 15.5 Medida de travessia do fogo

Uma pergunta importante é: **o fogo atravessou a floresta da esquerda para a direita?**

Aqui consideramos que houve travessia quando pelo menos uma célula da última coluna foi queimada ou está em chamas.

In [ ]:
def fire_crossed_grid(grid):
    right_edge = grid[:, -1]
    return bool(np.any((right_edge == FIRE) | (right_edge == BURNED)))

## 15.6 Simulação estendida completa

A função abaixo executa a versão estendida, registrando:

- árvores restantes;
- árvores em chamas;
- árvores queimadas;
- percentual queimado;
- vento sul e vento oeste a cada tick;
- ocorrência ou não de travessia.

In [ ]:
def run_simulation_extended(
    width=80,
    height=60,
    density=58,
    probability_of_spread=55,
    mean_moisture=0.25,
    moisture_variation=0.10,
    base_south_wind=10,
    base_west_wind=15,
    wind_amplitude=8,
    wind_noise=2,
    dynamic_wind_enabled=True,
    vegetation_probs=(0.35, 0.40, 0.25),
    big_jumps=True,
    max_ticks=300,
    seed=42,
    save_all_frames=False
):
    grid, vegetation, moisture, fire_age, initial_trees, rng = setup_extended(
        width=width,
        height=height,
        density=density,
        mean_moisture=mean_moisture,
        moisture_variation=moisture_variation,
        vegetation_probs=vegetation_probs,
        seed=seed
    )

    history = []
    frames = []
    snapshots = {}

    tick = 0

    while np.any(grid == FIRE) and tick < max_ticks:
        if dynamic_wind_enabled:
            south_wind, west_wind = dynamic_wind(
                tick,
                base_south=base_south_wind,
                base_west=base_west_wind,
                amplitude=wind_amplitude,
                noise=wind_noise,
                rng=rng
            )
        else:
            south_wind = base_south_wind
            west_wind = base_west_wind

        burned = int(np.sum(grid == BURNED))
        trees = int(np.sum(grid == TREE))
        burning = int(np.sum(grid == FIRE))

        history.append({
            "tick": tick,
            "trees": trees,
            "burning": burning,
            "burned": burned,
            "percent_burned": 100 * burned / max(initial_trees, 1),
            "south_wind": south_wind,
            "west_wind": west_wind,
            "crossed": fire_crossed_grid(grid)
        })

        if save_all_frames:
            frames.append(grid.copy())

        if tick in [0, 5, 10, 20, 40, 80, 120]:
            snapshots[tick] = grid.copy()

        grid, fire_age, ignitions = step_fire_extended(
            grid=grid,
            vegetation=vegetation,
            moisture=moisture,
            fire_age=fire_age,
            probability_of_spread=probability_of_spread,
            south_wind_speed=south_wind,
            west_wind_speed=west_wind,
            big_jumps=big_jumps,
            rng=rng
        )

        tick += 1

    final_burned = int(np.sum(grid == BURNED))
    final_trees = int(np.sum(grid == TREE))
    final_burning = int(np.sum(grid == FIRE))

    history.append({
        "tick": tick,
        "trees": final_trees,
        "burning": final_burning,
        "burned": final_burned,
        "percent_burned": 100 * final_burned / max(initial_trees, 1),
        "south_wind": np.nan,
        "west_wind": np.nan,
        "crossed": fire_crossed_grid(grid)
    })

    if save_all_frames:
        frames.append(grid.copy())

    snapshots[tick] = grid.copy()

    return {
        "grid": grid,
        "vegetation": vegetation,
        "moisture": moisture,
        "history": pd.DataFrame(history),
        "snapshots": snapshots,
        "frames": frames,
        "initial_trees": initial_trees,
        "crossed": fire_crossed_grid(grid)
    }

## 15.7 Executando um cenário estendido

Nesta execução, ativamos:

- umidade média;
- vento variável;
- vegetação heterogênea;
- saltos de faíscas.

In [ ]:
extended_result = run_simulation_extended(
    width=width,
    height=height,
    density=58,
    probability_of_spread=55,
    mean_moisture=0.30,
    moisture_variation=0.12,
    base_south_wind=10,
    base_west_wind=15,
    wind_amplitude=8,
    wind_noise=2,
    dynamic_wind_enabled=True,
    vegetation_probs=(0.30, 0.45, 0.25),
    big_jumps=True,
    max_ticks=300,
    seed=42,
    save_all_frames=True
)

extended_history = extended_result["history"]
extended_history.tail()

In [ ]:
plot_grid(extended_result["grid"], "Estado final — modelo estendido")

print("Fogo atravessou a grade?", extended_result["crossed"])
print("Árvores iniciais:", extended_result["initial_trees"])
print("Percentual queimado final:", round(extended_history["percent_burned"].iloc[-1], 2), "%")

## 15.8 Visualização da umidade e dos tipos de vegetação

A seguir, visualizamos a distribuição espacial da umidade e dos tipos de vegetação.

In [ ]:
plt.figure(figsize=(10, 7))
plt.imshow(extended_result["moisture"], vmin=0, vmax=1)
plt.title("Mapa de umidade da vegetação")
plt.colorbar(label="Umidade")
plt.axis("off")
plt.show()

In [ ]:
veg_cmap = ListedColormap(["white", "lightgreen", "olive", "darkgreen"])

plt.figure(figsize=(10, 7))
plt.imshow(extended_result["vegetation"], cmap=veg_cmap, vmin=0, vmax=3)
plt.title("Tipos de vegetação")
plt.axis("off")
plt.show()

## 15.9 Vento ao longo do tempo

Como o vento varia dinamicamente, podemos visualizar sua evolução temporal.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(extended_history["tick"], extended_history["south_wind"], label="Vento sul")
plt.plot(extended_history["tick"], extended_history["west_wind"], label="Vento oeste")
plt.xlabel("Tick")
plt.ylabel("Intensidade do vento")
plt.title("Variação temporal do vento")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 16. Repetições por cenário

Para pesquisa, uma única execução não é suficiente, pois o modelo possui aleatoriedade.

Agora vamos repetir a simulação várias vezes por cenário, variando a densidade e coletando estatísticas.

In [ ]:
def run_replicates_by_density(
    densities,
    n_replicates=20,
    width=80,
    height=60,
    probability_of_spread=55,
    mean_moisture=0.30,
    moisture_variation=0.12,
    base_south_wind=10,
    base_west_wind=15,
    wind_amplitude=8,
    wind_noise=2,
    dynamic_wind_enabled=True,
    vegetation_probs=(0.30, 0.45, 0.25),
    big_jumps=True,
    max_ticks=300,
    base_seed=1000
):
    rows = []

    for density_value in densities:
        for rep in range(n_replicates):
            seed = base_seed + int(density_value * 100) + rep

            result = run_simulation_extended(
                width=width,
                height=height,
                density=density_value,
                probability_of_spread=probability_of_spread,
                mean_moisture=mean_moisture,
                moisture_variation=moisture_variation,
                base_south_wind=base_south_wind,
                base_west_wind=base_west_wind,
                wind_amplitude=wind_amplitude,
                wind_noise=wind_noise,
                dynamic_wind_enabled=dynamic_wind_enabled,
                vegetation_probs=vegetation_probs,
                big_jumps=big_jumps,
                max_ticks=max_ticks,
                seed=seed,
                save_all_frames=False
            )

            h = result["history"]

            rows.append({
                "density": density_value,
                "replicate": rep + 1,
                "seed": seed,
                "initial_trees": result["initial_trees"],
                "final_percent_burned": float(h["percent_burned"].iloc[-1]),
                "ticks": int(h["tick"].max()),
                "crossed": result["crossed"],
                "final_trees": int(h["trees"].iloc[-1]),
                "final_burned": int(h["burned"].iloc[-1])
            })

    return pd.DataFrame(rows)

In [ ]:
densities_extended = [35, 45, 55, 65, 75]

replicates_df = run_replicates_by_density(
    densities=densities_extended,
    n_replicates=20,
    width=width,
    height=height,
    probability_of_spread=55,
    mean_moisture=0.30,
    moisture_variation=0.12,
    base_south_wind=10,
    base_west_wind=15,
    wind_amplitude=8,
    wind_noise=2,
    dynamic_wind_enabled=True,
    vegetation_probs=(0.30, 0.45, 0.25),
    big_jumps=True,
    max_ticks=300,
    base_seed=2026
)

replicates_df.head()

## 16.1 Média, desvio padrão e intervalo de confiança

O intervalo de confiança de 95% será calculado pela aproximação normal:

`IC95 = média ± 1,96 × erro padrão`

Essa aproximação é adequada para uma análise didática inicial. Para análises formais, pode-se usar bootstrap ou distribuição t de Student.

In [ ]:
summary_df = (
    replicates_df
    .groupby("density")
    .agg(
        n=("final_percent_burned", "count"),
        mean_percent_burned=("final_percent_burned", "mean"),
        std_percent_burned=("final_percent_burned", "std"),
        mean_ticks=("ticks", "mean"),
        crossing_rate=("crossed", "mean")
    )
    .reset_index()
)

summary_df["se_percent_burned"] = summary_df["std_percent_burned"] / np.sqrt(summary_df["n"])
summary_df["ci95_low"] = summary_df["mean_percent_burned"] - 1.96 * summary_df["se_percent_burned"]
summary_df["ci95_high"] = summary_df["mean_percent_burned"] + 1.96 * summary_df["se_percent_burned"]

summary_df

In [ ]:
plt.figure(figsize=(10, 6))
plt.errorbar(
    summary_df["density"],
    summary_df["mean_percent_burned"],
    yerr=1.96 * summary_df["se_percent_burned"],
    marker="o",
    capsize=5
)
plt.xlabel("Densidade inicial de árvores (%)")
plt.ylabel("Percentual médio queimado (%)")
plt.title("Percentual queimado médio por densidade — IC 95%")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(summary_df["density"], summary_df["crossing_rate"], marker="o")
plt.xlabel("Densidade inicial de árvores (%)")
plt.ylabel("Taxa de travessia do fogo")
plt.title("Probabilidade empírica de o fogo atravessar a grade")
plt.ylim(-0.05, 1.05)
plt.grid(True, alpha=0.3)
plt.show()

# 17. Exportação dos resultados em CSV

Os resultados das repetições e o resumo estatístico serão salvos em arquivos CSV.

Esses arquivos podem ser usados posteriormente em relatórios, artigos, Power BI, Excel ou análises estatísticas.

In [ ]:
from pathlib import Path

output_dir = Path("resultados_fire_simple_extension_3")
output_dir.mkdir(exist_ok=True)

replicates_csv = output_dir / "replicates_fire_extended.csv"
summary_csv = output_dir / "summary_fire_extended.csv"
history_csv = output_dir / "history_single_run_extended.csv"

replicates_df.to_csv(replicates_csv, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
extended_history.to_csv(history_csv, index=False, encoding="utf-8-sig")

print("Arquivos exportados:")
print(replicates_csv)
print(summary_csv)
print(history_csv)

# 18. Animação da propagação

A animação permite observar a dinâmica espacial do incêndio ao longo dos ticks.

No Jupyter Notebook, a animação pode ser exibida diretamente em HTML.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

frames = extended_result["frames"]

fig, ax = plt.subplots(figsize=(10, 7))
img = ax.imshow(frames[0], cmap=cmap, vmin=0, vmax=3)
ax.axis("off")
title = ax.set_title("Propagação do fogo — tick 0")

def update_animation(frame_index):
    img.set_data(frames[frame_index])
    title.set_text(f"Propagação do fogo — tick {frame_index}")
    return [img, title]

anim = FuncAnimation(
    fig,
    update_animation,
    frames=len(frames),
    interval=150,
    blit=False
)

plt.close(fig)

HTML(anim.to_jshtml())

## 18.1 Salvando a animação

A célula abaixo tenta salvar a animação em GIF.

Caso o ambiente não tenha suporte ao `pillow`, instale com:

```python
pip install pillow
```

In [ ]:
animation_path = output_dir / "animacao_fire_extended.gif"

try:
    anim.save(animation_path, writer="pillow", fps=8)
    print("Animação salva em:", animation_path)
except Exception as e:
    print("Não foi possível salvar a animação em GIF.")
    print("Erro:", e)

# 19. Conclusão da versão melhorada

A versão estendida permite transformar o modelo em um experimento computacional mais completo.

Com as melhorias adicionadas, é possível investigar:

- como a umidade reduz ou bloqueia a propagação;
- como ventos variáveis alteram a direção e a velocidade do fogo;
- como diferentes vegetações modificam a inflamabilidade;
- em quais condições o fogo atravessa toda a paisagem;
- como a aleatoriedade afeta os resultados;
- quais cenários apresentam maior risco médio;
- como comunicar os resultados por gráficos, tabelas, CSV e animações.

Essa estrutura é adequada para atividades de ensino, pesquisa exploratória e introdução à modelagem baseada em agentes.